# 🇹🇳🇮🇹 Générateur météo Tunisie + Italie — version sans accès Google Drive (solution à l’erreur d’authentification)

**Objectif :** générer deux CSV indépendants avec les **23 colonnes du fichier australien** `weatherAUS.csv`, à partir de données ERA5 réelles, sans installer Python.

### Mode recommandé pour le premier test
- **Tunisie : Sfax** ; **Italie : Rome** ; **2015–2025**.
- Les 12 villes sont configurées ; pour ajouter des villes, modifier `SELECTED_CITIES` dans la configuration.
- Le téléchargement se fait **une année à la fois** ; chaque année reçue est **mise en cache temporaire dans Colab**.
- Si Open-Meteo renvoie HTTP 429, **arrêter et reprendre ultérieurement**, en réimportant le ZIP de cache afin de ne pas retélécharger les années enregistrées. Le programme n'invente aucune donnée.
- Après exécution, fichiers `Tunisie_weather_23_columns.csv` et `Italie_weather_23_columns.csv`.

**Source :** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), réanalyse ERA5. Toutes les 23 colonnes ne correspondent pas exactement aux instruments de mesure du jeu australien ; les approximations sont documentées dans le fichier de méthodologie. Les séries de réanalyse ne sont pas des relevés de station garantis.

**Attention :** les mêmes 23 colonnes n'impliquent pas que l'ancien modèle australien fonctionne sur ces villes. Le notebook entraîne de nouveaux modèles par pays et évalue leurs résultats chronologiquement.

## 1. Configuration

Par défaut : **Sfax** pour la Tunisie et **Rome** pour l'Italie. C'est le moyen le plus raisonnable de vérifier la chaîne complète sans multiplier les grosses requêtes. Pour enrichir ensuite les CSV, ajoute des noms de villes à `SELECTED_CITIES`, puis relance le notebook : les années déjà téléchargées ne seront reprises que si le cache a été restauré dans la session Colab. Une requête HTTP 429 signifie qu'il faut respecter les limites de service et réessayer lorsque l'accès est rétabli.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time, zipfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('/content') if Path('/content').exists() else Path.cwd()
START_DATE, END_DATE = '2015-01-01', '2025-12-31'
RAIN_THRESHOLD_MM = 1.0
API = 'https://archive-api.open-meteo.com/v1/archive'
COUNTRIES = {
    'Tunisie': [
        ('Tunis', 36.8065, 10.1815, 'Africa/Tunis'),
        ('Sfax', 34.7406, 10.7603, 'Africa/Tunis'),
        ('Sousse', 35.8256, 10.63699, 'Africa/Tunis'),
        ('Bizerte', 37.2744, 9.8739, 'Africa/Tunis'),
        ('Gabes', 33.8815, 10.0982, 'Africa/Tunis'),
        ('Djerba', 33.8750, 10.8580, 'Africa/Tunis'),
    ],
    'Italie': [
        ('Rome', 41.9028, 12.4964, 'Europe/Rome'),
        ('Milan', 45.4642, 9.1900, 'Europe/Rome'),
        ('Turin', 45.0703, 7.6869, 'Europe/Rome'),
        ('Bologna', 44.4949, 11.3426, 'Europe/Rome'),
        ('Naples', 40.8518, 14.2681, 'Europe/Rome'),
        ('Palermo', 38.1157, 13.3615, 'Europe/Rome'),
    ],
}
AUS_COLUMNS = ['Date','Location','MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine',
               'WindGustDir','WindGustSpeed','WindDir9am','WindDir3pm',
               'WindSpeed9am','WindSpeed3pm','Humidity9am','Humidity3pm',
               'Pressure9am','Pressure3pm','Cloud9am','Cloud3pm','Temp9am','Temp3pm',
               'RainToday','RainTomorrow']
HOURLY = ['temperature_2m','relative_humidity_2m','pressure_msl','cloud_cover',
          'wind_speed_10m','wind_direction_10m','wind_gusts_10m']
DAILY = ['temperature_2m_min','temperature_2m_max','precipitation_sum',
         'et0_fao_evapotranspiration','sunshine_duration','wind_gusts_10m_max']
# Choisis les villes à télécharger. Pour le PREMIER TEST, garde UNE ville par pays.
# Une fois ce test réussi, ajoute les autres noms et conserve le ZIP de cache entre les sessions.
SELECTED_CITIES = {
    'Tunisie': ['Sfax'],    # Ensuite : 'Tunis', 'Sousse', 'Bizerte', 'Gabes', 'Djerba'
    'Italie': ['Rome'],     # Ensuite : 'Milan', 'Turin', 'Bologna', 'Naples', 'Palermo'
}
ALL_LOCATIONS = COUNTRIES
COUNTRIES = {
    country: [loc for loc in locations if loc[0] in SELECTED_CITIES[country]]
    for country, locations in ALL_LOCATIONS.items()
}
assert all(COUNTRIES.values()), 'Il faut au moins une ville par pays'

print('Format attendu :',len(AUS_COLUMNS),'colonnes identiques à weatherAUS.csv')
print('Période :',START_DATE,'au',END_DATE)
print('Villes par pays :', {k:len(v) for k,v in COUNTRIES.items()})

## 2. Télécharger les données météo — **sans Google Drive**

Cette version n'appelle pas `drive.mount()` : elle évite l'erreur `credential propagation was unsuccessful`.

- Les données reçues sont enregistrées provisoirement sous `/content/Weather_Tunisia_Italy_ERA5_cache`.
- Lors d'une erreur HTTP 429, le notebook tente de télécharger un ZIP de sauvegarde des années déjà reçues.
- **Le contenu de `/content` peut être effacé en fin de session Colab.** Conserve ton ZIP si tu souhaites reprendre plus tard.
- Ne relance pas des requêtes en boucle en cas de 429 (limites de service Open-Meteo).


In [ ]:
import gzip

# Version sans Google Drive : ne demande AUCUNE autorisation OAuth.
CACHE_DIR = BASE / 'Weather_Tunisia_Italy_ERA5_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Cache temporaire Colab :', CACHE_DIR)
print('IMPORTANT : télécharge le ZIP de cache si les requêtes échouent, car /content est éphémère.')

# Facultatif pour reprendre un téléchargement d'une session précédente :
# dépose le fichier weather_cache_partial.zip dans le panneau Fichiers de Colab,
# puis relance le notebook. S'il est présent, il est restauré automatiquement.
previous_cache_zip = BASE / 'weather_cache_partial.zip'
if previous_cache_zip.exists():
    restored = 0
    with zipfile.ZipFile(previous_cache_zip) as saved:
        for entry in saved.infolist():
            entry_path = Path(entry.filename)
            is_cache_file = (
                len(entry_path.parts) == 2
                and entry_path.parts[0] == CACHE_DIR.name
                and entry_path.suffixes[-2:] == ['.json', '.gz']
            )
            if not is_cache_file:
                continue
            out_file = CACHE_DIR / entry_path.name
            out_file.write_bytes(saved.read(entry))
            restored += 1
    print(f'✅ {restored} année(s) restaurée(s) depuis {previous_cache_zip.name}')

def backup_cache_download():
    files_to_save = sorted(CACHE_DIR.glob('*.json.gz'))
    if not files_to_save:
        print('Aucune année disponible pour la sauvegarde du cache.')
        return
    archive = BASE / 'weather_cache_partial.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as saved:
        for file in files_to_save:
            saved.write(file, 'Weather_Tunisia_Italy_ERA5_cache/' + file.name)
    print(f'Cache sauvegardé : {len(files_to_save)} année(s) → {archive.name}')
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print('Archive de cache :', archive)


SESSION = requests.Session()
CHUNKS = [(f'{year}-01-01',f'{year}-12-31') for year in range(2015,2026)]


def download_chunk(city, lat, lon, timezone_name, start, end):
    params = {
        'latitude':lat,'longitude':lon,'start_date':start,'end_date':end,
        'timezone':timezone_name,'models':'era5','cell_selection':'land',
        'hourly':','.join(HOURLY),'daily':','.join(DAILY),
        'temperature_unit':'celsius','wind_speed_unit':'kmh','precipitation_unit':'mm',
    }
    params_key = hashlib.sha256(json.dumps(params,sort_keys=True).encode()).hexdigest()[:12]
    filename = f'{city}_{start[:4]}_{params_key}.json.gz'
    cache_file = CACHE_DIR / filename
    if cache_file.exists():
        with gzip.open(cache_file,'rt',encoding='utf-8') as handle:
            obj=json.load(handle)
        return obj['payload'],obj['url'],True

    try:
        resp=SESSION.get(API,params=params,timeout=180)
    except requests.RequestException as exc:
        raise RuntimeError(f'Connexion impossible pour {city} {start} : {exc}. '
                           'Les années reçues restent dans le cache de la session.') from exc

    if resp.status_code == 429:
        retry=resp.headers.get('Retry-After')
        detail=f' (Retry-After: {retry})' if retry else ''
        raise RuntimeError(
            f'HTTP 429 pour {city}, année {start[:4]}{detail}. '
            'Quota ou concurrence temporairement limitée par Open-Meteo. '
            'Ne pas relancer immédiatement. Les années terminées sont sauvegardées temporairement; '
            'reprendre plus tard avec la même sélection.')
    resp.raise_for_status()
    payload=resp.json()
    if payload.get('error'):
        raise RuntimeError(f"Erreur Open-Meteo : {payload.get('reason','inconnue')}")
    for part,fields in [('hourly',HOURLY),('daily',DAILY)]:
        if part not in payload:
            raise ValueError(f'Partie {part} absente pour {city} {start}')
        missing=[name for name in fields if name not in payload[part]]
        if missing:
            raise ValueError(f'Variables {part} absentes pour {city} : {missing}')
    # Ecriture atomique pour ne pas laisser un cache JSON tronqué.
    tmp_path=cache_file.with_suffix('.tmp')
    with gzip.open(tmp_path,'wt',encoding='utf-8') as handle:
        json.dump({'payload':payload,'url':resp.url},handle)
    tmp_path.replace(cache_file)
    return payload,resp.url,False


def fetch_city(city, lat, lon, tz):
    hourly_frames,daily_frames,urls=[],[],[]
    for start,end in CHUNKS:
        raw,url,from_cache=download_chunk(city,lat,lon,tz,start,end)
        hourly_frames.append(pd.DataFrame(raw['hourly']))
        daily_frames.append(pd.DataFrame(raw['daily']))
        urls.append(url)
        print(f"  {'♻️ cache' if from_cache else '✅ API'} {city} {start[:4]}")
        if not from_cache:
            time.sleep(1)  # Pas de rafale de requêtes simultanées.
    hourly=pd.concat(hourly_frames,ignore_index=True)
    daily=pd.concat(daily_frames,ignore_index=True)
    if daily['time'].duplicated().any():
        raise ValueError(f'Dates journalières dupliquées : {city}')
    # Un passage à l'heure d'hiver peut répéter une heure civile en Italie.
    # Les mesures à 9h et 15h ne sont pas concernées, mais les doublons 2h sont possibles.
    for heure in (9,15):
        dates_heure = pd.to_datetime(hourly['time'])
        clefs = dates_heure.loc[dates_heure.dt.hour == heure].dt.strftime('%Y-%m-%d')
        if clefs.duplicated().any():
            raise ValueError(f'Doublon à {heure}h pour {city}')
    expected_dates=pd.date_range(START_DATE,END_DATE,freq='D').strftime('%Y-%m-%d')
    if list(daily['time']) != list(expected_dates):
        raise ValueError(f'Dates absentes ou non consécutives : {city}')
    print(f'✅ {city} : {len(daily):,} jours / {len(hourly):,} heures, série complète')
    return hourly,daily,urls

raw_by_country={}
request_urls={}
for country, locations in COUNTRIES.items():
    print(f'--- {country} ---')
    raw_by_country[country]={}
    for city,lat,lon,tz in locations:
        try:
            hourly,daily,urls=fetch_city(city,lat,lon,tz)
        except Exception:
            backup_cache_download()
            raise
        raw_by_country[country][city]=(hourly,daily)
        request_urls[city]=urls
print('✅ Téléchargement terminé pour les villes sélectionnées :',SELECTED_CITIES)


## 3. Produire les **23 colonnes compatibles en structure**

Même ordre de colonnes que `weatherAUS.csv`. Les approximations sont signalées dans la méthodologie : `Evaporation` est la référence ET₀ (et non une mesure d'évaporation), `WindGustDir` une direction estimée et les nuages sont convertis en octas. La cible `RainTomorrow` est calculée sur **le lendemain réel dans la même ville**. Les lignes manquantes ne sont jamais inventées.

In [ ]:
COMPASS=['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW']
def cardinal(deg):
    return np.nan if pd.isna(deg) else COMPASS[int(((float(deg)%360)+11.25)//22.5)%16]
def oktas(percent):
    return np.nan if pd.isna(percent) else int(np.clip(np.rint(float(percent)*8/100),0,8))
def flag_rain(mm):
    return np.nan if pd.isna(mm) else ('Yes' if float(mm)>RAIN_THRESHOLD_MM else 'No')

def convert_station(city, hourly, daily):
    h=hourly.copy(); d=daily.copy()
    d['Date']=pd.to_datetime(d['time']).dt.strftime('%Y-%m-%d')
    h['dt']=pd.to_datetime(h['time'])
    h['Date']=h['dt'].dt.strftime('%Y-%m-%d')
    h['hour']=h['dt'].dt.hour
    h9=h[h.hour.eq(9)].set_index('Date')
    h15=h[h.hour.eq(15)].set_index('Date')
    gust=h.sort_values(['Date','wind_gusts_10m'],ascending=[True,False],na_position='last')
    gust=gust.drop_duplicates('Date').set_index('Date')
    dates=d['Date']
    def lookup(frame,field):
        return pd.to_numeric(dates.map(frame[field]),errors='coerce')
    rows=pd.DataFrame({
        'Date':dates, 'Location':city,
        'MinTemp':d['temperature_2m_min'], 'MaxTemp':d['temperature_2m_max'],
        'Rainfall':d['precipitation_sum'],
        'Evaporation':d['et0_fao_evapotranspiration'], # Proxy ET0; pas une evaporation mesuree
        'Sunshine':pd.to_numeric(d['sunshine_duration'],errors='coerce')/3600,
        'WindGustDir':lookup(gust,'wind_direction_10m').apply(cardinal), # Proxy
        'WindGustSpeed':d['wind_gusts_10m_max'],
        'WindDir9am':lookup(h9,'wind_direction_10m').apply(cardinal),
        'WindDir3pm':lookup(h15,'wind_direction_10m').apply(cardinal),
        'WindSpeed9am':lookup(h9,'wind_speed_10m'),
        'WindSpeed3pm':lookup(h15,'wind_speed_10m'),
        'Humidity9am':lookup(h9,'relative_humidity_2m'),
        'Humidity3pm':lookup(h15,'relative_humidity_2m'),
        'Pressure9am':lookup(h9,'pressure_msl'),
        'Pressure3pm':lookup(h15,'pressure_msl'),
        'Cloud9am':lookup(h9,'cloud_cover').apply(oktas),
        'Cloud3pm':lookup(h15,'cloud_cover').apply(oktas),
        'Temp9am':lookup(h9,'temperature_2m'),
        'Temp3pm':lookup(h15,'temperature_2m'),
    }).sort_values('Date').reset_index(drop=True)
    rows['RainToday']=rows['Rainfall'].apply(flag_rain)
    rows['RainTomorrow']=rows['RainToday'].shift(-1)
    rows=rows.iloc[:-1].copy()
    for col in rows.columns:
        if col not in ['Date','Location','WindGustDir','WindDir9am','WindDir3pm','RainToday','RainTomorrow']:
            rows[col]=pd.to_numeric(rows[col],errors='coerce').round(2)
    return rows[AUS_COLUMNS]

country_datasets={}
for country,places in COUNTRIES.items():
    frames=[]
    for city,_,_,_ in places:
        h,d=raw_by_country[country][city]
        frames.append(convert_station(city,h,d))
    frame=pd.concat(frames,ignore_index=True)
    assert list(frame.columns)==AUS_COLUMNS
    assert not frame.duplicated(['Date','Location']).any()
    assert frame['RainTomorrow'].notna().all(), f'RainTomorrow manquant pour {country}'
    assert frame['RainTomorrow'].isin(['No','Yes']).all()
    assert len(frame)>3000*len(places), 'Trop peu de jours; vérifier les années'
    assert not frame[['MinTemp','MaxTemp','Rainfall']].isna().any().any(),f'Données essentielles manquantes pour {country}'
    country_datasets[country]=frame
    filename=f'{country}_weather_23_columns.csv'
    frame.to_csv(BASE/filename,index=False,encoding='utf-8',float_format='%.2f')
    print(f'✅ {filename} : {len(frame):,} lignes × {len(frame.columns)} colonnes')
    display(frame.head(2))
print('✅ DEUX CSV créés dans',BASE)

## 4. Télécharger les **deux CSV**

Cette cellule crée `Tunisie_weather_23_columns.csv` et `Italie_weather_23_columns.csv`, ainsi que `METHODOLOGIE_ET_SOURCES.json`, réunis dans un ZIP. **Les CSV comprennent uniquement les villes sélectionnées** dans la section 1. Pour obtenir les 12 villes, compléter `SELECTED_CITIES` progressivement et reprendre les nouvelles années à partir du cache sauvegardé et réimporté.

In [ ]:
meta={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source':'Open-Meteo Historical Weather API / ERA5 reanalysis (modélisation, non relevés de station)',
    'source_url':'https://open-meteo.com/en/docs/historical-weather-api',
    'ERA5_url':'https://climate.copernicus.eu/climate-reanalysis',
    'attribution':'Weather data by Open-Meteo.com (CC BY 4.0)',
    'period_start':START_DATE,'period_end':END_DATE,
    'selected_cities': SELECTED_CITIES, 'available_cities':{k:[place[0] for place in v] for k,v in ALL_LOCATIONS.items()},
    'columns':AUS_COLUMNS,'rain_rule':'RainToday = Yes si precipitation_sum > 1.0 mm ; RainTomorrow = RainToday du lendemain pour la même ville',
    'columns_not_direct_equivalents':{
        'Evaporation':'ET0 FAO evapotranspiration de référence mm/j, PAS l evaporation mesuree',
        'WindGustDir':'Direction instantanee du vent a l heure du maximum de rafales horaire, PAS direction verifiee de la rafale',
        'Cloud9am/Cloud3pm':'Conversion estimee du couvert nuageux en % vers octas 0-8',
        'Sunshine':'Duree quotidienne d ensoleillement modelisee (secondes / 3600 heures)',
    },
    'files':{},'api_urls':request_urls,
}
for country,df_country in country_datasets.items():
    f=BASE/f'{country}_weather_23_columns.csv'
    meta['files'][f.name]={
        'rows':len(df_country),
        'cities':{k:int(v) for k,v in df_country['Location'].value_counts().items()},
        'rain_tomorrow_rate':float((df_country['RainTomorrow']=='Yes').mean()),
        'missing_by_column':{k:int(v) for k,v in df_country.isna().sum().items()},
        'sha256':hashlib.sha256(f.read_bytes()).hexdigest(),
    }
metadata_file=BASE/'METHODOLOGIE_ET_SOURCES.json'
metadata_file.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
zip_file=BASE/'Tunisie_Italie_2_Datasets_Meteo.zip'
with zipfile.ZipFile(zip_file,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(metadata_file,metadata_file.name)
    for country in COUNTRIES:
        f=BASE/f'{country}_weather_23_columns.csv'
        z.write(f,f.name)
print('Deux CSV indépendants dans le ZIP :')
for country in COUNTRIES:
    print(' •',f'{country}_weather_23_columns.csv')
print('ZIP :',zip_file)
try:
    from google.colab import files
    files.download(str(zip_file))
except ImportError:
    print('Pour télécharger hors Colab, ouvrir:',zip_file)

## 5. Vérifications de qualité

Contrôle des 23 noms de colonnes, doublons date/ville, période, valeurs manquantes, fréquence de pluie et villes sélectionnées. Ce contrôle doit passer pour chaque pays avant de publier le dataset.

In [ ]:
for country,df_country in country_datasets.items():
    print('===',country,'===')
    print('Colonnes identiques au fichier AUS :',list(df_country.columns)==AUS_COLUMNS)
    print('Doublons date/ville :',int(df_country.duplicated(['Date','Location']).sum()))
    print('Pluie demain (%):',round(100*(df_country['RainTomorrow']=='Yes').mean(),2))
    print('Lignes sans valeurs manquantes :',df_country.dropna().shape[0], '/',len(df_country))
    display(df_country.groupby('Location').agg(Date_debut=('Date','min'),Date_fin=('Date','max'),Jours=('Date','size')))
    print('Manquants par champ:',df_country.isna().sum()[lambda x:x>0].to_dict())

## 6. Machine Learning : modèles **séparés** Tunisie / Italie

Comparer la régression logistique et Random Forest avec entraînement **jusqu'en 2023** et test **2024–2025** (séparation temporelle). Il faut vérifier notamment le rappel (recall), la précision et ROC-AUC ; l'accuracy seule est insuffisante. Le modèle australien existant ne doit pas être prétendu valide en Tunisie ou en Italie sans adaptation.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

trained_models={}
all_results=[]
for country,df_country in country_datasets.items():
    df=df_country.copy()
    df['Season']=pd.to_datetime(df['Date']).dt.month.map(lambda m: 'Winter' if m in (12,1,2) else 'Spring' if m in (3,4,5) else 'Summer' if m in (6,7,8) else 'Autumn')
    cols=[c for c in df.columns if c not in ('Date','RainTomorrow')]
    train=df['Date']<'2023-12-31'
    test=df['Date']>='2024-01-01'
    Xtr,ytr=df.loc[train,cols],df.loc[train,'RainTomorrow']
    Xte,yte=df.loc[test,cols],df.loc[test,'RainTomorrow']
    numeric=Xtr.select_dtypes(include='number').columns.tolist()
    cats=[k for k in cols if k not in numeric]
    def pipe(classifier):
        prep=ColumnTransformer([
            ('num',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
            ('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cats),
        ])
        return Pipeline([('preprocessor',prep),('classifier',classifier)])
    candidates={
        'Logistic Regression':pipe(LogisticRegression(max_iter=1500,random_state=42)),
        'Random Forest':pipe(RandomForestClassifier(n_estimators=150,min_samples_leaf=2,random_state=42,n_jobs=-1)),
    }
    print('====',country,'====')
    print('Train avant 2024:',len(Xtr),'| Test 2024–2025:',len(Xte))
    print('Baseline sans pluie (test):', round((yte == 'No').mean(), 3))
    assert 'Date' not in cols and 'RainTomorrow' not in cols, 'Fuite de cible détectée'
    assert df.loc[train,'Date'].max() == '2023-12-30'
    assert df.loc[test,'Date'].min() == '2024-01-01'
    assert ytr.nunique()==2 and yte.nunique()==2,'Classes insuffisantes pour évaluer'
    for name,model in candidates.items():
        model.fit(Xtr,ytr)
        pred=model.predict(Xte)
        score=model.predict_proba(Xte)[:,list(model.classes_).index('Yes')]
        row={
            'Country':country,'Model':name,'Accuracy':accuracy_score(yte,pred),
            'Precision':precision_score(yte,pred,pos_label='Yes',zero_division=0),
            'Recall':recall_score(yte,pred,pos_label='Yes',zero_division=0),
            'F1':f1_score(yte,pred,pos_label='Yes',zero_division=0),
            'ROC-AUC':roc_auc_score((yte=='Yes').astype(int),score),
        }
        all_results.append(row)
        trained_models[(country,name)]=model
    display(pd.DataFrame([r for r in all_results if r['Country']==country]).round(3))

scores=pd.DataFrame(all_results)
scores.to_csv(BASE/'Tunisie_Italie_ML_resultats.csv',index=False)
display(scores.round(3))
for country in COUNTRIES:
    subset=scores.loc[scores.Country==country]
    ax=subset.set_index('Model')[['Accuracy','Precision','Recall','F1','ROC-AUC']].T.plot.bar(figsize=(8,4),rot=0)
    ax.set_title(f'{country} — Evaluation chronologique 2024–2025')
    ax.set_ylim(0,1)
    ax.set_ylabel('Score')
    plt.tight_layout()
    plt.show()

## 7. Tester 5 prédictions réelles par pays

Après l'entraînement ci-dessus, ce bloc donne *Pluie réelle*, *Prédiction* et *Probabilité de pluie*. Les cinq exemples sont descriptifs et ne remplacent pas les métriques globales.

In [ ]:
for country,frame in country_datasets.items():
    test=frame['Date']>='2024-01-01'
    sub=frame.loc[test].head(5).copy()
    sub['Season']=pd.to_datetime(sub['Date']).dt.month.map(lambda m:'Winter' if m in (12,1,2) else 'Spring' if m in (3,4,5) else 'Summer' if m in (6,7,8) else 'Autumn')
    model=trained_models[(country,'Random Forest')]
    X=sub.drop(columns=['Date','RainTomorrow'])
    preds=model.predict(X)
    probs=model.predict_proba(X)[:,list(model.classes_).index('Yes')]
    print('===',country,'===')
    display(pd.DataFrame({'Date':sub['Date'].to_numpy(),'Ville':sub['Location'].to_numpy(),
                          'Pluie réelle':sub['RainTomorrow'].to_numpy(),
                          'Prédiction':preds,'Probabilité de pluie (%)':np.round(probs*100,2)}))

## 8. Résultats et limites

- Les **CSV** correspondent aux mêmes noms/ordre de 23 colonnes que le fichier australien ; leur **source est différente** et certains champs sont des **proxies**, décrits dans le fichier JSON.
- Les données ERA5 sont des **réanalyses** (observations + modèles), et non une garantie de mesures météorologiques directes en station.
- Le niveau de performance sur une ville exclue de l'entraînement, la stabilité climatique, la calibration des probabilités et une vraie prévision en temps réel **n'ont pas été vérifiés**.
- Pour GitHub, publie le notebook exécuté, les deux CSV si leur licence l'autorise, le fichier de traçabilité, et les captures obtenues pendant l'exécution ; ne réutilise pas le score australien comme score tunisien ou italien.
- Sources : [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) et [ERA5 — Copernicus](https://climate.copernicus.eu/climate-reanalysis). Attribution : **Weather data by Open-Meteo.com (CC BY 4.0)** ; vérifie les conditions d'utilisation selon ton contexte.
- **Sans Drive** : le cache est temporaire. En cas d’échec le programme propose un téléchargement ZIP ; les CSV finaux sont téléchargés séparément à l’étape 4.
